
# TEL351 - Tarea 1: Agentes periodistas y generación automatizada de noticias

Nombre: Pedro Arce.

## Objetivo
En esta tarea desarrollarás un sistema que scrapea noticias de un sitio web público, crea agentes automatizados que generan nuevas versiones de estas noticias, y luego presenta el resultado en un sitio web HTML.

## Detalle

- Recolectar al menos 100 noticias de un sitio web de libre acceso utilizando web scraping.

- Procesar los datos obtenidos y almacenarlos de manera estructurada.

- Diseñar 5 agentes, cada uno con una personalidad distinta, que generen noticias basadas en las originales o en su interpretación.

- Construir un sitio web HTML estático que simule ser un portal de noticias con las publicaciones generadas por los agentes.

## Parte 1: Web Scraping de noticias

In [1]:

# Importar librerías necesarias
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time



### Instrucciones:
- Elige un sitio de noticias público (ej. [BioBioChile](https://www.biobiochile.cl/), [Emol](https://www.emol.com/), [CNN Chile](https://www.cnnchile.com/), [BBC Mundo](https://www.bbc.com/mundo)) que:
    - No requiera login.
    - Tenga estructura consistente (mismo layout para artículos).
    - Permita scraping ético (consulta `robots.txt` si es necesario).
- Usa Python con `requests` + `BeautifulSoup`, o bien con `selenium` si es un sitio dinámico.
- Extrae y almacena para **al menos 100 artículos**:
    - Título.
    - Fecha.
    - Cuerpo completo de la noticia.
    - URL original.
    - Categoría o sección (si es posible).
- Guarda los datos en un archivo `.json`, `.csv` o `.pkl` estructurado.
- Asegúrate de limpiar caracteres extraños, eliminar etiquetas HTML y normalizar el texto (acentos, espacios, etc).

In [2]:
###1. Configuración y utilidades

import re
import json
import unicodedata
from urllib.parse import urlparse
from datetime import datetime
import random

#Sitio objetivo (CNN Chile)
BASE = "https://www.cnnchile.com"
ROBOTS = f"{BASE}/robots.txt"
SITEMAP_INDEX = f"{BASE}/_files/sitemaps/sitemap_index.xml"
SITEMAP_NEWS  = f"{BASE}/_files/sitemaps/sitemap_news.xml"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; ProyScraper/1.0; +https://tu-dominio-ejemplo.cl/)",
    "Accept-Language": "es,es-CL;q=0.9,en;q=0.8"
}

CATEGORY_WHITELIST = {
    "pais","mundo","negocios","cultura","deportes","servicios","bits","miradas","opinion","cnn-data","futuro-360"
}

def fetch(url, retries=3, backoff=2.0, timeout=15):
    """DESCARGA ROBUSTA → devuelve BYTES (no texto)."""
    for i in range(retries):
        try:
            r = requests.get(url, headers=HEADERS, timeout=timeout)
            if 200 <= r.status_code < 300:
                return r.content  #clave para evitar mojibake
            time.sleep(backoff * (i + 1))
        except requests.RequestException:
            time.sleep(backoff * (i + 1))
    return None

def normalize_text(s: str) -> str:
    if not s:
        return ""
    s = unicodedata.normalize("NFKC", s)
    s = s.replace("\xa0", " ").replace("\u200b", "")
    s = re.sub(r"\s+", " ", s, flags=re.M)
    return s.strip()

def is_article_url(url: str) -> bool:
    try:
        path = urlparse(url).path.strip("/")
        first = path.split("/", 1)[0].lower()
        if first in CATEGORY_WHITELIST:
            return True
        if first in {"tag","category","programas","en-vivo","search"}:
            return False
        return False
    except Exception:
        return False

In [3]:
###2. Sitemaps => recolectar URLs (>= 100)

def parse_xml_locs(xml_bytes) -> list:
    """Extrae <loc> desde XML (recibe BYTES)."""
    if not xml_bytes:
        return []
    soup = BeautifulSoup(xml_bytes, "xml")
    return [loc.get_text(strip=True) for loc in soup.find_all("loc")]

def get_sitemap_months():
    xml = fetch(SITEMAP_INDEX)
    months = parse_xml_locs(xml)
    months = [m for m in months if "/_files/sitemaps/" in m]
    months.sort(reverse=True)  #mas recientes primero
    return months

def get_news_urls(limit=200):
    urls = []

    news_xml = fetch(SITEMAP_NEWS)
    urls.extend(parse_xml_locs(news_xml))

    if len(urls) < limit:
        for sm in get_sitemap_months():
            xml = fetch(sm)
            urls.extend(parse_xml_locs(xml))
            if len(urls) >= limit * 2:
                break

    seen = set()
    filtered = []
    for u in urls:
        if not u or not u.startswith("http"):
            continue
        if is_article_url(u) and u not in seen:
            seen.add(u)
            filtered.append(u)

    filtered = [u for u in filtered if not re.search(r"\.(jpg|jpeg|png|webp)(\?|$)", u, re.I)]
    return filtered[:max(limit, 120)]

article_urls = get_news_urls(limit=200)
len(article_urls), article_urls[:5]

(179,
 ['https://www.cnnchile.com/negocios/travel-sale-2025-como-aprovechar-descuentos-sin-caer-en-estafas_20250825/',
  'https://www.cnnchile.com/pais/caso-convenios-defensa-daniel-andrade-pide-reabrir-investigacion-democracia-viva-apunta-acusacion-pruebas_20250825/',
  'https://www.cnnchile.com/negocios/travel-sale-2025-chile-5-consejos-clave-para-aprovechar-las-mejores-ofertas-de-viajes_20250825/',
  'https://www.cnnchile.com/cultura/concierto-big-time-rush-chile-2026-entradas_20250825/',
  'https://www.cnnchile.com/pais/robo-homicidio-curacavi-por-que-fiscalia-investigando-rol-alcalde-hernandez-detencion-imputados_20250825/'])

In [4]:
###3. Parser de artículos (título, fecha, cuerpo, url, categoría)

DATE_PATTERNS = [
    ("%Y-%m-%dT%H:%M:%S%z", r"\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}(?:Z|[+-]\d{2}:\d{2})"),
    ("%d.%m.%Y / %H:%M", r"\b\d{2}\.\d{2}\.\d{4}\s*/\s*\d{2}:\d{2}\b"),
]

BAD_PREFIXES = (
    "Lee también", "LEA TAMBIÉN", "Relacionado", "RELACIONADO", 
    "Video:", "VIDEO:", "Vea también", "VEA TAMBIÉN",
    "Compartir", "DESTACAMOS", "LO ÚLTIMO", "PROGRAMAS DE TV"
)

def parse_date_from_meta(soup: BeautifulSoup) -> str | None:
    metas = [
        ("meta", {"property": "article:published_time"}),
        ("meta", {"name": "article:published_time"}),
        ("meta", {"property": "og:updated_time"}),
        ("meta", {"name": "date"}),
        ("meta", {"itemprop": "datePublished"}),
        ("time",  {"itemprop": "datePublished"}),
    ]
    for tag_name, attrs in metas:
        tag = soup.find(tag_name, attrs=attrs)
        if tag:
            content = tag.get("content") or tag.get_text(strip=True)
            if content:
                iso = content.replace("Z", "+00:00")
                try:
                    return datetime.fromisoformat(iso).isoformat()
                except Exception:
                    pass
    return None

def parse_date_fallback(page_text: str) -> str | None:
    for fmt, regex in DATE_PATTERNS:
        m = re.search(regex, page_text)
        if m:
            date_str = m.group(0)
            try:
                dt = datetime.strptime(date_str, fmt)
                return dt.isoformat()
            except Exception:
                continue
    return None

def get_category_from_url(url: str) -> str:
    path = urlparse(url).path.strip("/")
    return (path.split("/", 1)[0] if path else "").lower()

def extract_main_paragraphs(soup: BeautifulSoup) -> list[str]:
    candidates = []
    for tag in soup.find_all(["article", "section", "div"]):
        ps = tag.find_all("p")
        if len(ps) >= 3:
            candidates.append((len(ps), tag))
    candidates.sort(key=lambda x: x[0], reverse=True)

    paragraphs = []
    if candidates:
        _, best = candidates[0]
        for p in best.find_all("p"):
            txt = normalize_text(p.get_text(" ", strip=True))
            if not txt:
                continue
            if any(txt.startswith(bad) for bad in BAD_PREFIXES):
                continue
            if len(txt) < 25 and re.search(r"^([A-ZÁÉÍÓÚÑ# ]{3,}|[A-ZÁÉÍÓÚÑ0-9 ]{3,})$", txt):
                continue
            paragraphs.append(txt)
    return paragraphs

def parse_article(url: str) -> dict | None:
    html_bytes = fetch(url)
    if not html_bytes:
        return None

    soup = BeautifulSoup(html_bytes, "html.parser")

    h1 = soup.find("h1")
    title = normalize_text(h1.get_text(strip=True)) if h1 else ""

    date_iso = parse_date_from_meta(soup)
    if not date_iso:
        page_text = soup.get_text("\n", strip=True)
        date_iso = parse_date_fallback(page_text)

    body_paragraphs = extract_main_paragraphs(soup)
    body = normalize_text(" ".join(body_paragraphs))

    categoria = get_category_from_url(url)

    #Fallback JSON-LD para completar título/cuerpo/fecha
    if not title or not body or len(body) < 200:
        for script in soup.find_all("script", type="application/ld+json"):
            try:
                data = json.loads(script.string or "")
                items = [data] if isinstance(data, dict) else (data if isinstance(data, list) else [])
                for item in items:
                    if isinstance(item, dict) and item.get("@type") in {"NewsArticle","Article"}:
                        title = title or normalize_text(item.get("headline",""))
                        body = body or normalize_text(item.get("articleBody",""))
                        if not date_iso and item.get("datePublished"):
                            try:
                                iso = item["datePublished"].replace("Z","+00:00")
                                date_iso = datetime.fromisoformat(iso).isoformat()
                            except Exception:
                                pass
            except Exception:
                continue

    if not title or not body or len(body) < 200:
        return None

    return {
        "titulo": title,
        "fecha": date_iso or "",
        "cuerpo": body,
        "url": url,
        "categoria": categoria,
    }

In [5]:
###4. Ejecutar scraping (>=100), validar y guardar

def polite_sleep():
    time.sleep(0.7 + random.random() * 0.5)

records = []
for i, url in enumerate(article_urls):
    art = parse_article(url)
    if art:
        records.append(art)
    if (i+1) % 20 == 0:
        print(f"Procesadas {i+1} URLs, válidas: {len(records)}")
    polite_sleep()
    if len(records) >= 130:  #sobrecolectar
        break

df = pd.DataFrame.from_records(records)
df = df.drop_duplicates(subset=["url"]).dropna(subset=["titulo","cuerpo"])
print(df.shape)
df.head(3)

Procesadas 20 URLs, válidas: 20
Procesadas 40 URLs, válidas: 40
Procesadas 60 URLs, válidas: 60
Procesadas 80 URLs, válidas: 80
Procesadas 100 URLs, válidas: 100
Procesadas 120 URLs, válidas: 120
(130, 5)


,titulo,fecha,cuerpo,url,categoria
0,Travel Sale 2025: ¿Cómo aprovechar descuentos ...,2025-08-25T17:17:00,La Asociación Gremial de Marcas Retail entregó...,https://www.cnnchile.com/negocios/travel-sale-...,negocios
1,Caso Convenios: Defensa de Daniel Andrade pide...,2025-08-25T17:05:00,"La petición, presentada ante el Juzgado de Gar...",https://www.cnnchile.com/pais/caso-convenios-d...,pais
2,Travel Sale 2025 Chile: 5 consejos clave para ...,2025-08-25T16:08:00,¡Ya comenzó la segunda versión del Travel Sale...,https://www.cnnchile.com/negocios/travel-sale-...,negocios


In [6]:
###5. Limpieza final y exportación

df["titulo"]   = df["titulo"].apply(normalize_text)
df["cuerpo"]   = df["cuerpo"].apply(normalize_text)
df["categoria"] = df["categoria"].str.lower().str.strip()

#Filtro de seguridad
df = df[df["cuerpo"].str.len() >= 400].copy()
df.reset_index(drop=True, inplace=True)

print("Total artículos listos:", len(df))
assert len(df) >= 100, "No se alcanzaron 100 artículos. Amplía el rango o relaja filtros."

CSV_OUT  = "cnnchile_noticias.csv"
JSON_OUT = "cnnchile_noticias.json"

#BOM para compatibilidad Excel/Windows
df.to_csv(CSV_OUT, index=False, encoding="utf-8-sig")
df.to_json(JSON_OUT, orient="records", force_ascii=False)

CSV_OUT, JSON_OUT

Total artículos listos: 129


('cnnchile_noticias.csv', 'cnnchile_noticias.json')

## Parte 2: Diseño de Agentes Generadores de Contenido


### Instrucciones:
- Diseña **5 agentes con personalidades distintas**.
- Cada agente debe generar al menos 10 noticias basadas en el corpus extraído.
- Cada agente debe tener un comportamiento **automatizado y único** al generar contenido. Por ejemplo:

| Agente                 | Personalidad           | Comportamiento esperado                                                                                                                           |
| ---------------------- | ---------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------- |
| 1. Aleatorio           | "Publicador impulsivo" | Elige noticias al azar, publica un extracto sin cambios.                                                                                          |
| 2. Imitador            | "Papagayo"             | Copia noticias, pero cambia algunas palabras por sinónimos usando NLP básico.                                                                     |
| 3. Seleccionador       | "Curador temático"     | Filtra noticias de una temática y publica resúmenes cortos.                                                                                       |
| 4. Reescritor creativo | "Periodista junior"    | Reescribe noticias con un estilo personal (con reglas simples de cambio de redacción).                                                            |
| 5. Generador semántico | "Periodista LLM"       | Usa un modelo de lenguaje (`transformers`, `GPT`, `LLaMA`, etc.) para redactar nuevas versiones de las noticias originales, interpretándolas. |


In [7]:
# Define cada agente

import pandas as pd

agentes = [
    {
        "Agente": "PrimerParrafo",
        "Personalidad": "Literal básico",
        "Comportamiento esperado": "Publica el título y el PRIMER párrafo del cuerpo sin cambios (solo limpieza)."
    },
    {
        "Agente": "DiezMasNuevas",
        "Personalidad": "Bot puntual",
        "Comportamiento esperado": "Toma las 10 noticias más recientes por fecha y publica un extracto de 40–60 palabras del inicio."
    },
    {
        "Agente": "Recorte120",
        "Personalidad": "Editor de extensión",
        "Comportamiento esperado": "Publica el título y un recorte del cuerpo a 120 palabras exactas, sin reescritura ni opinión."
    },
    {
        "Agente": "PorCategoriaSimple",
        "Personalidad": "Curador de sección",
        "Comportamiento esperado": "Filtra por una categoría fija (p.ej., 'mundo') y publica las DOS primeras oraciones del cuerpo."
    },
    {
        "Agente": "TagsClave",
        "Personalidad": "Bot de palabras clave",
        "Comportamiento esperado": "Extrae 5 palabras frecuentes (sin stopwords) y publica: título + primera oración + lista de #tags."
    },
]

agentes_df = pd.DataFrame(agentes, columns=["Agente","Personalidad","Comportamiento esperado"])
agentes_df

,Agente,Personalidad,Comportamiento esperado
0,PrimerParrafo,Literal básico,Publica el título y el PRIMER párrafo del cuer...
1,DiezMasNuevas,Bot puntual,Toma las 10 noticias más recientes por fecha y...
2,Recorte120,Editor de extensión,Publica el título y un recorte del cuerpo a 12...
3,PorCategoriaSimple,Curador de sección,"Filtra por una categoría fija (p.ej., 'mundo')..."
4,TagsClave,Bot de palabras clave,Extrae 5 palabras frecuentes (sin stopwords) y...


In [8]:
# Ejecución de cada agente sobre el corpus

###1. Cargar corpus

import os

if "df" not in globals():
    if os.path.exists("cnnchile_noticias.json"):
        df = pd.read_json("cnnchile_noticias.json")
    elif os.path.exists("cnnchile_noticias.csv"):
        df = pd.read_csv("cnnchile_noticias.csv")
    else:
        raise FileNotFoundError("No se encuentra df ni archivos cnnchile_noticias.(json|csv). Ejecuta la Parte 1 primero.")

#Validaciones minimas
required_cols = {"titulo","fecha","cuerpo","url","categoria"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Faltan columnas en df: {missing}")

#Limpieza básica defensiva
df = df.dropna(subset=["titulo","cuerpo","url"]).copy()
df["categoria"] = df["categoria"].astype(str).str.strip().str.lower()
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
df = df.reset_index(drop=True)

print("Artículos cargados:", len(df))
df.head(2)

Artículos cargados: 129


,titulo,fecha,cuerpo,url,categoria
0,Travel Sale 2025: ¿Cómo aprovechar descuentos ...,2025-08-25 17:17:00,La Asociación Gremial de Marcas Retail entregó...,https://www.cnnchile.com/negocios/travel-sale-...,negocios
1,Caso Convenios: Defensa de Daniel Andrade pide...,2025-08-25 17:05:00,"La petición, presentada ante el Juzgado de Gar...",https://www.cnnchile.com/pais/caso-convenios-d...,pais


In [9]:
###2. Utilidades (tokenización, oraciones, normalizacion, tags)

from collections import Counter

def normalize_spaces(s: str) -> str:
    s = (s or "").replace("\xa0"," ").replace("\u200b"," ")
    s = re.sub(r"\s+", " ", s, flags=re.M)
    return s.strip()

def split_sentences_es(text: str):
    text = normalize_spaces(text)
    #Cortes por . ? ! seguidos de espacio/mayúscula o fin
    #Mantiene los signos en la oracion
    parts = re.split(r'(?<=[\.\?\!])\s+(?=[A-ZÁÉÍÓÚÜÑ])', text)
    #Filtro de vacios
    return [p.strip() for p in parts if p and len(p.strip()) > 0]

def first_paragraph_or_two_sentences(cuerpo: str) -> str:
    if "\n\n" in cuerpo:
        para = cuerpo.split("\n\n", 1)[0]
        return normalize_spaces(para)
    sents = split_sentences_es(cuerpo)
    return normalize_spaces(" ".join(sents[:2])) if sents else ""

def first_n_words(text: str, n: int) -> str:
    tokens = re.findall(r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ0-9]+", text)
    if len(tokens) <= n:
        return normalize_spaces(text)
    return normalize_spaces(" ".join(tokens[:n])) + "…"

#Stopwords básicas de español
STOP_ES = {
    "a","ante","bajo","cabe","con","contra","de","desde","durante","en","entre","hacia","hasta","mediante",
    "para","por","según","sin","so","sobre","tras","y","o","u","e","ni","que","como","al","del","un","una",
    "unos","unas","el","la","los","las","su","sus","lo","le","les","se","ya","no","sí","más","menos","este",
    "esta","estos","estas","ese","esa","eso","esos","esas","aquel","aquella","aquello","aquellos","aquellas",
    "ser","estar","haber","hacer","puede","pueden","fue","fueron","son","eran","es","era"
}

def top_k_words(text: str, k: int = 5):
    toks = re.findall(r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]{2,}", text.lower())
    toks = [t for t in toks if t not in STOP_ES]
    cnt = Counter(toks)
    return [w for w,_ in cnt.most_common(k)]

In [10]:
###3. Definición de agentes

import numpy as np

def agente_primer_parrafo(df, n=10, seed=42):
    rng = np.random.default_rng(seed)
    idx = rng.choice(df.index, size=min(n, len(df)), replace=False)
    rows = []
    for i in idx:
        r = df.loc[i]
        salida = first_paragraph_or_two_sentences(r["cuerpo"])
        rows.append({
            "agente": "PrimerParrafo",
            "titulo": r["titulo"],
            "contenido": salida,  #primer parrafo (o 2 oraciones)
            "url": r["url"],
            "categoria": r["categoria"],
            "fecha": r["fecha"]
        })
    return pd.DataFrame(rows)

def agente_diez_mas_nuevas(df, n=10):
    tmp = df.dropna(subset=["fecha"]).sort_values("fecha", ascending=False).head(n)
    rows = []
    for _, r in tmp.iterrows():
        extracto = first_n_words(r["cuerpo"], 60)  #~40–60 palabras
        rows.append({
            "agente": "DiezMasNuevas",
            "titulo": r["titulo"],
            "contenido": extracto,
            "url": r["url"],
            "categoria": r["categoria"],
            "fecha": r["fecha"]
        })
    #si faltan por NaT, completar aleatorio
    if len(rows) < n:
        faltan = n - len(rows)
        extra = df.sample(faltan, random_state=1)
        for _, r in extra.iterrows():
            extracto = first_n_words(r["cuerpo"], 60)
            rows.append({
                "agente": "DiezMasNuevas",
                "titulo": r["titulo"],
                "contenido": extracto,
                "url": r["url"],
                "categoria": r["categoria"],
                "fecha": r["fecha"]
            })
    return pd.DataFrame(rows).head(n)

def agente_recorte_120(df, n=10, seed=7):
    rng = np.random.default_rng(seed)
    idx = rng.choice(df.index, size=min(n, len(df)), replace=False)
    rows = []
    for i in idx:
        r = df.loc[i]
        recorte = first_n_words(r["cuerpo"], 120)  #120 palabras exactas (o menos si el cuerpo es corto)
        rows.append({
            "agente": "Recorte120",
            "titulo": r["titulo"],
            "contenido": recorte,
            "url": r["url"],
            "categoria": r["categoria"],
            "fecha": r["fecha"]
        })
    return pd.DataFrame(rows)

def agente_por_categoria_simple(df, categoria_pref="mundo", n=10):
    dff = df[df["categoria"] == categoria_pref]
    if len(dff) < n:
        #fallback: usar la categoría mas frecuente
        top_cat = df["categoria"].value_counts().index[0]
        dff = df[df["categoria"] == top_cat]
    dff = dff.head(n)
    rows = []
    for _, r in dff.iterrows():
        sents = split_sentences_es(r["cuerpo"])
        contenido = " ".join(sents[:2]) if sents else r["cuerpo"]
        rows.append({
            "agente": "PorCategoriaSimple",
            "titulo": r["titulo"],
            "contenido": normalize_spaces(contenido),
            "url": r["url"],
            "categoria": r["categoria"],
            "fecha": r["fecha"]
        })
    return pd.DataFrame(rows)

def agente_tags_clave(df, n=10, seed=19):
    rng = np.random.default_rng(seed)
    idx = rng.choice(df.index, size=min(n, len(df)), replace=False)
    rows = []
    for i in idx:
        r = df.loc[i]
        sents = split_sentences_es(r["cuerpo"])
        primera = sents[0] if sents else first_n_words(r["cuerpo"], 30)
        tags = top_k_words(r["cuerpo"], 5)
        tags_fmt = " ".join(f"#{t}" for t in tags)
        contenido = f"{primera} {tags_fmt}".strip()
        rows.append({
            "agente": "TagsClave",
            "titulo": r["titulo"],
            "contenido": normalize_spaces(contenido),
            "url": r["url"],
            "categoria": r["categoria"],
            "fecha": r["fecha"]
        })
    return pd.DataFrame(rows)

In [11]:
###4. Ejecutar cada agente

primer_parrafo_df   = agente_primer_parrafo(df, n=10)
diez_mas_nuevas_df  = agente_diez_mas_nuevas(df, n=10)
recorte_120_df      = agente_recorte_120(df, n=10)
por_categoria_df    = agente_por_categoria_simple(df, categoria_pref="mundo", n=10)
tags_clave_df       = agente_tags_clave(df, n=10)

#Unir todo (util para la parte del portal HTML)
agentes_out = pd.concat([
    primer_parrafo_df,
    diez_mas_nuevas_df,
    recorte_120_df,
    por_categoria_df,
    tags_clave_df
], ignore_index=True)

print("Total publicaciones generadas:", len(agentes_out))
agentes_out.head(10)

Total publicaciones generadas: 50


,agente,titulo,contenido,url,categoria,fecha
0,PrimerParrafo,Codelco informa que aporte al fisco creció 24%...,Desde la empresa también indicaron que en los ...,https://www.cnnchile.com/negocios/codelco-apor...,negocios,2025-08-22 14:19:00
1,PrimerParrafo,India suspende envíos postales a EE. UU. por f...,El Departamento de Correos indio paralizó temp...,https://www.cnnchile.com/pais/india-suspende-e...,pais,2025-08-23 12:14:00
2,PrimerParrafo,Gobierno acusa retroceso en propuesta previsio...,La vocera Camila Vallejo y el ministro del Tra...,https://www.cnnchile.com/pais/gobierno-critica...,pais,2025-08-25 13:23:00
3,PrimerParrafo,Investigan intoxicación de estudiante por cons...,La menor de primer año medio fue trasladada ur...,https://www.cnnchile.com/pais/investigan-intox...,pais,2025-08-23 16:08:00
4,PrimerParrafo,Grau asegura que la responsabilidad fiscal ser...,El nuevo ministro destacó que el compromiso co...,https://www.cnnchile.com/pais/grau-asegura-que...,pais,2025-08-24 13:54:00
5,PrimerParrafo,Elizalde asegura que Argentina y Chile colabor...,El ministro del Interior también reconoció el ...,https://www.cnnchile.com/pais/elizalde-asegura...,pais,2025-08-22 14:40:00
6,PrimerParrafo,En medio de choque masivo: Carabineros frustra...,Uniformados detuvieron a dos antisociales que ...,https://www.cnnchile.com/pais/en-medio-de-choq...,pais,2025-08-23 13:28:00
7,PrimerParrafo,"Alejandro “Mono” González, referente chileno d...","El fundador de la Brigada Ramona Parra, Gonzál...",https://www.cnnchile.com/cultura/alejandro-mon...,cultura,2025-08-25 12:30:00
8,PrimerParrafo,Última semana de restricción vehicular en Sant...,Hasta el 31 de agosto rige la medida que busca...,https://www.cnnchile.com/pais/restriccion-vehi...,pais,2025-08-25 08:14:00
9,PrimerParrafo,La Piojera se despide de Santiago: Negociacion...,"El administrador de La Piojera, Mauricio Gajar...",https://www.cnnchile.com/pais/la-piojera-se-de...,pais,2025-08-22 21:22:00


In [12]:
###5. Guardar salidas por agente y combinado

#Archivos por agente
primer_parrafo_df.to_csv("agente_PrimerParrafo.csv", index=False, encoding="utf-8-sig")
diez_mas_nuevas_df.to_csv("agente_DiezMasNuevas.csv", index=False, encoding="utf-8-sig")
recorte_120_df.to_csv("agente_Recorte120.csv", index=False, encoding="utf-8-sig")
por_categoria_df.to_csv("agente_PorCategoriaSimple.csv", index=False, encoding="utf-8-sig")
tags_clave_df.to_csv("agente_TagsClave.csv", index=False, encoding="utf-8-sig")

#Combinado (util para las plantillas HTML)
agentes_out.to_csv("agentes_publicaciones.csv", index=False, encoding="utf-8-sig")
agentes_out.to_json("agentes_publicaciones.json", orient="records", force_ascii=False)

## Parte 3: Generación de portal de noticias en HTML


### Instrucciones:

- Crea un sitio web HTML básico (una sola página o varias secciones).
- Usa los textos generados por los agentes para crear un portal de noticias HTML.
- Muestra las noticias generadas por tus agentes, agrupadas por autor (agente).
- Incluye:
    - Título.
    - Fecha.
    - Nombre del agente.
    - Cuerpo del texto generado.
    - Imágenes (opcional).
- Puedes estilizar con CSS básico si lo deseas, pero se evalúa el contenido funcional.
- Puedes generar el HTML directamente desde Python con `Jinja2`, o usando templates simples.

In [13]:
# Código para generar HTML con noticias agrupadas por agente

## Reflexión final


Crea un archivo `README.md` y responde:
- ¿Cuál agente generó contenido más coherente?
- ¿Qué dificultades tuviste al automatizar la generación de noticias?
- ¿Qué herramientas o métodos usarías si lo hicieras de nuevo?


## Entregables

Un archivo comprimido `.zip` con nombre `T1_TEL351_Nombre_Apellido.zip` que contenga:

1. **Jupyter Notebook** con todo el proceso:

    - Scraping

    - Procesamiento de datos

    - Implementación de los agentes

    - Generación del sitio

2. **Carpeta** `datos/` con el corpus original y el generado por los agentes.

3. **Carpeta** `sitio/` con archivos HTML y medios asociados.

4. **Archivo** `README.md`